In [ ]:
# Auto-install helper: ensure required packages are available (runs in notebook)
# Installs only the missing package 'openai' to resolve ModuleNotFoundError seen when importing.
import sys
import subprocess
import pkgutil

def ensure_install(pkg_spec: str, pkg_name: str = None):
    """Install a package if it is not already importable.
    pkg_spec: pip install spec (e.g., 'openai>=1.0.0')
    pkg_name: import name (defaults to first token of pkg_spec)
    """
    if pkg_name is None:
        pkg_name = pkg_spec.split("[", 1)[0].split("==", 1)[0].split(">=", 1)[0].split("<", 1)[0]
    if pkgutil.find_spec(pkg_name) is None:
        print(f"Instalando paquete: {pkg_spec} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg_spec])
    else:
        print(f"Paquete '{pkg_name}' ya está instalado.")

# Try to import openai and install if missing
try:
    import openai
    print("✓ 'openai' importado correctamente.")
except Exception:
    ensure_install("openai>=1.0.0", "openai")
    import importlib
    importlib.invalidate_caches()
    try:
        import openai
        print("✓ 'openai' instalado e importado correctamente.")
    except Exception as e:
        print(" No se pudo importar 'openai' después de la instalación:", e)
        print("Intenta ejecutar en una terminal: pip install -r requirements.txt")


AttributeError: module 'pkgutil' has no attribute 'find_spec'

In [ ]:


import os
import time
from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory


os.environ["GITHUB_TOKEN"] = os.getenv("GITHUB_TOKEN", "tu_token_aqui")
os.environ["OPENAI_BASE_URL"] = "https://models.inference.ai.azure.com" 

print(f"✓ Entorno configurado para proyecto Camanchaca")


def inicializar_modelos():
    try:
  
        llm_analitico = ChatOpenAI(
            base_url=os.getenv("OPENAI_BASE_URL"),
            api_key=os.getenv("GITHUB_TOKEN"),
            model="gpt-4o",
            temperature=0.1,
            streaming=True
        )
        
       
        llm_ligero = ChatOpenAI(
            base_url=os.getenv("OPENAI_BASE_URL"),
            api_key=os.getenv("GITHUB_TOKEN"),
            model="gpt-4o-mini",
            temperature=0.5
        )
        return llm_analitico, llm_ligero
    except Exception as e:
        print(f" Error en inicialización: {e}")
        return None, None

llm_camanchaca, llm_resumen = inicializar_modelos()


store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

def gestionar_resumen(session_id: str, llm):
    history = get_session_history(session_id)
  
    if len(history.messages) > 6:
        print("\n--- [Sistema: Generando resumen de contexto] ---")
        messages_to_summarize = history.messages[:-2]
        content = "\n".join([f"{m.type}: {m.content}" for m in messages_to_summarize])
        
        resumen = llm.invoke(f"Resume los puntos clave sobre la producción de salmones discutidos aquí:\n{content}")
        
        recent_messages = history.messages[-2:]
        history.clear()
        history.add_ai_message(f"Resumen anterior: {resumen.content}")
        history.messages.extend(recent_messages)


prompt = ChatPromptTemplate.from_messages([
    ("system", """Eres un experto en acuicultura y analista de datos para Salmones Camanchaca en Puerto Montt. 
    Tu objetivo es ayudar a optimizar la producción, salud de peces y logística. 
    Responde de forma técnica pero accesible."""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

chain = prompt | llm_camanchaca

asistente_final = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)


def chat_camanchaca():
    session_id = "vuelo_camanchaca_01"
    print("\n=== SISTEMA IA CAMANCHACA ACTIVO ===")
    print("Escribe 'salir' para finalizar.\n")

    while True:
        user_input = input(" Usuario: ")
        if user_input.lower() in ['salir', 'exit']: break
        
  
        gestionar_resumen(session_id, llm_resumen)
        
        print(" Asistente: ", end="", flush=True)
        
        try:
           
            for chunk in asistente_final.stream(
                {"input": user_input},
                config={"configurable": {"session_id": session_id}}
            ):
                print(chunk.content, end="", flush=True)
            print("\n")
            
        except Exception as e:
            print(f"\n Error en flujo: {e}")

if __name__ == "__main__":
    chat_camanchaca()

ModuleNotFoundError: No module named 'openai'